# 3D Packing Problem

#### Importing packages

In [64]:
import numpy as np 
import pandas as pd
from amplpy import AMPL

In [65]:
pd.set_option('display.max_rows', 500)

### Notations

- $I=\{1,2,...,N\}$ is the set of SKUs, where N denotes the total number of SKUs to be packed.

- $J=\{1,2,...,M\}$ is the set of DFCs, M denotes the number of different types of DFCs available for packing.
  
- $T_{j}$ is the set of DFCs of type j, $\forall  j \in J$

- The $i^{th}$ SKU is characterized by its length $l_i$, width $w_i$, height $h_i$, weight $m_i$. $\forall i \in I$

- The $j^{th}$ DFC is characterized by its length $L_j$, width $W_j$, height $H_j$ and maximum weight carrying capacity $G_j$. $\forall j \in J$

- $lr^+_{ii′} = 1$, if $i^{th}$ SKU is placed on left-hand side of $i′^{th}$ SKU; otherwise, $lr^+_{ii′} = 0$.
- $bf^+_{ii′} = 1$, if $i^{th}$ SKU is placed behind the $i′^{th}$ SKU;    otherwise, $bf^+_{ii′} = 0$.
- $bt^+_{ii′} = 1$, if $i^{th}$ SKU is placed on below the $i′^{th}$ SKU; otherwise, $bt^+_{ii′} = 0$.
- $lr^-_{ii′} = 1$, if $i^{th}$ SKU is placed on right-hand side of $i′^{th}$  SKU; otherwise, $lr^-_{ii′} = 0$.
- $bf^-_{ii′} = 1$, if $i^{th}$ SKU is placed in front of $i′^{th}$ SKU; otherwise, $bf^-_{ii′} = 0$.
- $bt^-_{ii′} = 1$, if $i^{th}$ SKU is placed above the $i′^{th}$ SKU; otherwise, $bt^-_{ii′} = 0$.

- $u_{jk}$ is a binary variable, which denotes whether $k_{th}$ copy of the $j_{th}$ DFC is being used for packaging.

- $s_{ijk}$ is a binary variable to denote, whether the $i_{th}$ SKU is packed in $k_{th}$ copy of
the $j_{th}$ DFC.

- The variables $x_i$, $y_i$, $z_i$ denotes the coodinated of the left-bottom-back corner of
$i_{th}$ SKU.

- The Variables $l^x_i$ , $l^y_i$ , $l^z_i$ denotes whether the length edge of ith SKU is parallel to X-axis, Y-axis or Z-axis.

- Similarly, the variables $w^x_i$ , $w^y_i$ , $w^z_i$ denotes whether the width edge of ith SKU is parallel to X-axis, Y-axis or Z-axis, and the variables $h^x_i$ , $h^y_i$ , $h^z_i$ denotes whether the height edge of $i_{th}$ SKU is parallel to X-axis, Y-axis or Z-axis.


### Formulation

The objective is to minimize the total vacant spaces or unused volume within
the assigned DFCs or, equivalently, the total volume of the chosen DFCs for
packing. The latter is easier to model. The overall model is as follows: 

$$
\begin{align}

\min \quad 
& \sum_{j\in J}\sum_{k\in T_j} u_{jk} L_j W_j H_j 
- \sum_{i\in I} l_i w_i h_i \\

\text{s.t.} \quad 
& \sum_{j\in J}\sum_{k\in T_j} s_{ijk} = 1 
&& \forall i \in I \\

& u_{jk} \ge s_{ijk}
&& \forall i \in I,\; \forall j \in J,\; \forall k \in T_j \\

& l_i^x + l_i^y + l_i^z = 1
&& \forall i \in I \\

& w_i^x + w_i^y + w_i^z = 1
&& \forall i \in I \\

& h_i^x + h_i^y + h_i^z = 1
&& \forall i \in I \\

& l_i^x + w_i^x + h_i^x = 1
&& \forall i \in I \\

& l_i^y + w_i^y + h_i^y = 1
&& \forall i \in I \\

& l_i^z + w_i^z + h_i^z = 1
&& \forall i \in I \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le L_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le W_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le H_j + (1-s_{ijk})M
&& \forall i\in I,\forall j\in J,\forall k\in T_j \\

& \sum_{i\in I} s_{ijk} m_i \le G_j
&& \forall j\in J,\forall k\in T_j \\

& x_i + l_i^x l_i + w_i^x w_i + h_i^x h_i
\le x_{i'} + (1-lr_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& x_{i'} + l_{i'}^x l_{i'} + w_{i'}^x w_{i'} + h_{i'}^x h_{i'}
\le x_i + (1-lr_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& y_i + l_i^y l_i + w_i^y w_i + h_i^y h_i
\le y_{i'} + (1-bf_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& y_{i'} + l_{i'}^y l_{i'} + w_{i'}^y w_{i'} + h_{i'}^y h_{i'}
\le y_i + (1-bf_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& z_i + l_i^z l_i + w_i^z w_i + h_i^z h_i
\le z_{i'} + (1-bt_{ii'}^{+})M
&& i<i',\forall i,i'\in I \\

& z_{i'} + l_{i'}^z l_{i'} + w_{i'}^z w_{i'} + h_{i'}^z h_{i'}
\le z_i + (1-bt_{ii'}^{-})M
&& i<i',\forall i,i'\in I \\

& lr_{ii'}^{+} + lr_{ii'}^{-}
+ bf_{ii'}^{+} + bf_{ii'}^{-}
+ bt_{ii'}^{+} + bt_{ii'}^{-}
\ge s_{ijk} + s_{i'jk} - 1
&& i<i',\forall i,i'\in I,\forall j\in J,\forall k\in T_j

\end{align}
$$


#### Adding constraint so that the lowest indexed copy of dfc is used first(but not used here)
$$
\begin{align}
u_{jp} \ge u_{jq}
\qquad
\forall j \in J,\;
p,q \in T_j: q = p + 1,\;
\end{align}
$$

- Just add the code below in model function before solve to apply the constraint:


    ampl.eval(
        """
        s.t. c20 {j in DFC, p in Copy[j], q in Copy[j] : q = p+1}:
        copy_used[j,p] >= copy_used[j,q];
        """
        )

fix command to fix a solution 

In [66]:
def model(a):
    with pd.ExcelFile(a) as xls:
        dfc_df = pd.read_excel(xls,"DFC")
        sku_df = pd.read_excel(xls, "SKU")

    print('dfc', dfc_df.to_string(), sep='\n')
    print('sku', sku_df.to_string(), sep='\n')


    # Initialize ampl object
    ampl = AMPL()

    # Make model
    ampl.eval(
        """
        set SKU;             
        set DFC;              
        set Copy{DFC};                            # copies of j_th DFC 
        set axis= {'x', 'y', 'z'};
        set dim= {'Length', 'Width', 'Height'};

        param dim_sku {SKU, dim};
        param sku_Weight {SKU};
        param dim_dfc {DFC, dim};
        param dfc_Weight {DFC};
        param M;       

        var relative_position_left {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_right {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_back {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_front {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_below {{i in SKU, l in SKU : i<l}} binary;
        var relative_position_above {{i in SKU, l in SKU : i<l}} binary;

        var copy_used {j in DFC, k in Copy[j]} binary;
        var sku_in_copy {i in SKU, j in DFC, k in Copy[j]} binary;
        var sku_position {SKU, axis} >=0;

        var Length_orientation {SKU, axis} binary;
        var Width_orientation {SKU, axis} binary;
        var Height_orientation {SKU, axis} binary;

        minimize vacant_space:
        sum{j in DFC} sum{k in Copy[j]} copy_used[j,k] *dim_dfc[j,'Length']*dim_dfc[j,'Width']*dim_dfc[j,'Height'] - sum{i in SKU} dim_sku[i,'Length']*dim_sku[i,'Width']*dim_sku[i,'Height']; 

        s.t. c1 {i in SKU}: 
        sum{j in DFC} sum{k in Copy[j]} sku_in_copy[i,j,k] = 1;

        s.t. c2 {i in SKU}:
        Length_orientation[i,'x'] + Length_orientation[i,'y'] + Length_orientation[i,'z'] = 1;

        s.t. c3 {i in SKU}:
        Width_orientation[i,'x'] + Width_orientation[i,'y'] + Width_orientation[i,'z'] = 1;

        s.t. c4 {i in SKU}:
        Height_orientation[i,'x'] + Height_orientation[i,'y'] + Height_orientation[i,'z'] = 1;

        s.t. c5 {i in SKU}:
        Length_orientation[i,'x'] + Width_orientation[i,'x'] + Height_orientation[i,'x'] = 1;

        s.t. c6 {i in SKU}:
        Length_orientation[i,'y'] + Width_orientation[i,'y'] + Height_orientation[i,'y'] = 1;

        s.t. c7 {i in SKU}:
        Length_orientation[i,'z'] + Width_orientation[i,'z'] + Height_orientation[i,'z'] = 1;

        s.t. c8 {i in SKU, j in DFC, k in Copy[j]}:  
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= dim_dfc[j, 'Length'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c9 {i in SKU, j in DFC, k in Copy[j]}: 
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= dim_dfc[j, 'Width'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c10 {i in SKU, j in DFC, k in Copy[j]}: 
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= dim_dfc[j, 'Height'] + (1- sku_in_copy[i,j,k])*M;
        
        s.t. c11 {i in SKU, j in DFC, k in Copy[j]}: 
        copy_used[j,k] >= sku_in_copy[i,j,k];

        s.t. c12 {j in DFC, k in Copy[j]}:
        sum{i in SKU} sku_in_copy[i,j,k]*sku_Weight[i] <= dfc_Weight[j];

        s.t. c13 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'x'] + Length_orientation[i,'x']*dim_sku[i,'Length'] + Width_orientation[i,'x']*dim_sku[i,'Width'] + Height_orientation[i,'x']*dim_sku[i,'Height'] <= sku_position[l,'x'] + (1- relative_position_left[i,l])*M;
        
        s.t. c14 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'x'] + Length_orientation[l,'x']*dim_sku[l,'Length'] + Width_orientation[l,'x']*dim_sku[l,'Width'] + Height_orientation[l,'x']*dim_sku[l,'Height'] <= sku_position[i,'x'] + (1- relative_position_right[i,l])*M;

        s.t. c15 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'y'] + Length_orientation[i,'y']*dim_sku[i,'Length'] + Width_orientation[i,'y']*dim_sku[i,'Width'] + Height_orientation[i,'y']*dim_sku[i,'Height'] <= sku_position[l,'y'] + (1- relative_position_back[i,l])*M;
        
        s.t. c16 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'y'] + Length_orientation[l,'y']*dim_sku[l,'Length'] + Width_orientation[l,'y']*dim_sku[l,'Width'] + Height_orientation[l,'y']*dim_sku[l,'Height'] <= sku_position[i,'y'] + (1- relative_position_front[i,l])*M;

        s.t. c17 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[i,'z'] + Length_orientation[i,'z']*dim_sku[i,'Length'] + Width_orientation[i,'z']*dim_sku[i,'Width'] + Height_orientation[i,'z']*dim_sku[i,'Height'] <= sku_position[l,'z'] + (1- relative_position_below[i,l])*M;
        
        s.t. c18 {(i,l) in {i in SKU, l in SKU : i<l}}:
        sku_position[l,'z'] + Length_orientation[l,'z']*dim_sku[l,'Length'] + Width_orientation[l,'z']*dim_sku[l,'Width'] + Height_orientation[l,'z']*dim_sku[l,'Height'] <= sku_position[i,'z'] + (1- relative_position_above[i,l])*M;
        
        s.t. c19 {i in SKU, l in SKU, j in DFC, k in Copy[j] : i<l}:
        relative_position_left[i,l] + relative_position_right[i,l] + relative_position_back[i,l] + relative_position_front[i,l] + relative_position_below[i,l] + relative_position_above[i,l] >= sku_in_copy[i,j,k] + sku_in_copy[l,j,k] -1;

        """
        )



    # Feeding data to the model
    ampl.set['SKU'] = sku_df['sku'].tolist()
    ampl.set['DFC'] = dfc_df['Type'].tolist()
    ampl.set['Copy'] = {
        row['Type']: list(range(1, row['Number of Copies'] + 1))
        for _, row in dfc_df.iterrows()
    }

    ampl.param['dim_sku'] = sku_df.set_index('sku')[['Length', 'Width', 'Height']]
    ampl.param['sku_Weight'] = sku_df.set_index('sku')['Weight']
    ampl.param['dim_dfc'] = dfc_df.set_index('Type')[['Length', 'Width', 'Height']]
    ampl.param['dfc_Weight'] = dfc_df.set_index('Type')['Weight']

    # FIXED: M = largest DFC dimension
    ampl.param['M'] = float(dfc_df[['Length', 'Width', 'Height']].values.max())


    # solve
    ampl.option['solver'] = 'gurobi'
    ampl.option['timelimit'] = 300  # give it more time for larger instances
    ampl.solve()

    solve_result = ampl.get_value('solve_result')
    print(f"Solve result: {solve_result}")
    print(f"Objective:    {ampl.get_value('vacant_space')}")

    if 'infeasible' in solve_result:
        # --- IIS block: only runs when broken ---
        ampl.option["gurobi_options"] = "iisfind=1"
        ampl.solve()

        from amplpy import OutputHandler

        class CaptureOutput(OutputHandler):
            def __init__(self): self.lines = []
            def output(self, kind, msg): self.lines.append(msg)

        capture = CaptureOutput()
        ampl.set_output_handler(capture)
        ampl.eval("""
            for {j in 1.._ncons} {
                if _con[j].iis != 'non' then
                    printf "CON | %-6s | %s\\n", _con[j].iis, _conname[j];
            }
            for {j in 1.._nvars} {
                if _var[j].iis != 'non' then
                    printf "VAR | %-6s | %s\\n", _var[j].iis, _varname[j];
            }
        """)
        ampl.set_output_handler(None)

        
        records = []
        for line in capture.lines:
            line = line.strip()
            if line.startswith("CON |") or line.startswith("VAR |"):
                parts = [p.strip() for p in line.split("|")]
                records.append({"kind": parts[0], "iis": parts[1], "name": parts[2]})

        if records:
            df = pd.DataFrame(records)
            df["group"] = df["name"].str.replace(r"\[.*\]", "", regex=True)
            print("\n=== IIS Members ===")
            print(df.to_string(index=False))
            print("\n=== IIS by constraint group ===")
            print(df.groupby(["kind","group","iis"]).size()
                    .reset_index(name="count").to_string(index=False))

    else:
        # --- Results block: only runs when solved ---
        copy_used_df     = ampl.var['copy_used'].get_values().to_pandas()
        sku_in_copy_df   = ampl.var['sku_in_copy'].get_values().to_pandas()
        sku_position_df  = ampl.var['sku_position'].get_values().to_pandas()
        Length_orientation_df = ampl.var['Length_orientation'].get_values().to_pandas()
        Width_orientation_df = ampl.var['Width_orientation'].get_values().to_pandas()
        Height_orientation_df = ampl.var['Height_orientation'].get_values().to_pandas()

        # copies used per DFC type from solver output
        copies_used_df = copy_used_df[copy_used_df['copy_used.val'] > 0.5].reset_index()
        copies_used_df.columns = ['dfc_type', 'copy', 'val']
        copies_per_type = copies_used_df.groupby('dfc_type').size().reset_index(name='copies_used')

        # volume per type straight from original dfc_df (not from solver variable)
        dfc_df['dfc_volume_each'] = dfc_df['Length'] * dfc_df['Width'] * dfc_df['Height']

        # merge on Type
        result = copies_per_type.merge(
            dfc_df[['Type', 'dfc_volume_each']],
            left_on='dfc_type',
            right_on='Type'
        )

        result['dfc_volume_total'] = result['copies_used'] * result['dfc_volume_each']

        dfc_volume_total = result['dfc_volume_total'].sum()

        # vacant_space = total DFC volume used − total SKU volume (your objective)
        # so packing efficiency = 1 - vacant / total_dfc
        packing_efficiency = 1 - ampl.get_value('vacant_space') / dfc_volume_total

        # Gurobi reports its own solve time via this suffix
        gurobi_time = ampl.get_value('_total_solve_time')
        print(f"Solve time (Gurobi): {gurobi_time:.4f} seconds")

        print(f"Packing efficiency: {packing_efficiency * 100:.2f}%")

        print("\n=== Copies used ===")
        print(copy_used_df[copy_used_df['copy_used.val'] > 0.5])

        print("\n=== SKU assignments ===")
        print(sku_in_copy_df[sku_in_copy_df['sku_in_copy.val'] > 0.5])

        print("\n=== SKU positions ===")
        print(sku_position_df)
        
        print("\n=== Length orientation ===")
        print(Length_orientation_df[Length_orientation_df['Length_orientation.val'] > 0.5])

        print("\n=== Width orientation ===")
        print(Width_orientation_df[Width_orientation_df['Width_orientation.val'] > 0.5])

        print("\n=== Height orientation ===")
        print(Height_orientation_df[Height_orientation_df['Height_orientation.val'] > 0.5])

#### Edge cases 
- an sku that can fit in only one particular dfc

In [67]:
model('3dpack1.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 4       3      3       3       4
1     2                 5       4      4       4       4
2     3                 2       5      5       5       5
sku
   sku  Length  Width  Height  Weight
0    1       3      3       3       3
1    2       3      3       3       3
2    3       4      4       3       3
3    4       3      3       3       3
4    5       3      4       4       3
5    6       2      3       3       3
6    7       5      5       5       5
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 41
13 simplex iterations
1 branching node
Solve result: solved
Objective:    41
Solve time (Gurobi): 0.0938 seconds
Packing efficiency: 88.64%

=== Copies used ===
               copy_used.val
index0 index1               
1      1                   1
       2                   1
       3                   1
       4                   1
2      1                   1
       2                   1
3      

- a set of sku which can fit in one particular dfc in one particular orientation

In [68]:
model('3dpack2.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       3      4       2       5
1     2                 3       5      6       4       7
sku
   sku  Length  Width  Height  Weight
0    1       2      3       4       5
1    2       6      4       5       7
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 0
0 simplex iterations
1 branching node
Solve result: solved
Objective:    0
Solve time (Gurobi): 0.0469 seconds
Packing efficiency: 100.00%

=== Copies used ===
               copy_used.val
index0 index1               
1      2                   1
2      1                   1

=== SKU assignments ===
                      sku_in_copy.val
index0 index1 index2                 
1      1      2                     1
2      2      1                     1

=== SKU positions ===
               sku_position.val
index0 index1                  
1      x                      0
       y                      0
       z                      0
2      x    

- an unfitable sku


In [69]:
model('3dpack3.xlsx')

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       4      4       4      16
1     2                 3       2      2       2       4
sku
   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
6    7       4      4       5       7
Gurobi 12.0.3:Gurobi 12.0.3: infeasible problem
0 simplex iterations

suffix dunbdd OUT;
Solve result: infeasible
Objective:    -114
Gurobi 12.0.3:   alg:iisfind = 1
Gurobi 12.0.3: infeasible problem
0 simplex iterations

suffix iis symbolic OUT;

option iis_table '\
0	non	not in the iis\
1	low	lower bound in the iis\
2	fix	both bounds in the iis\
3	upp	upper bound in the iis\
4	mem	member\
5	pmem	possible member\
6	plow	possibly lower bound\
7	pupp	possibly upper bound\
8	bug\
';

=== IIS Members ===
kind iis  

- adding one sku occupies one more dfc

In [70]:
model('3dpack4.xlsx')

from io import BytesIO

# Read the original Excel file into memory
with open('3dpack4.xlsx', 'rb') as f:
    file_bytes = BytesIO(f.read())

# Load the sheet you want
df = pd.read_excel(file_bytes, sheet_name='SKU')
df1 = pd.read_excel(file_bytes, sheet_name='DFC')

# Add a new row
new_row = pd.DataFrame([{'sku': 7, 'Length': 3, 'Width': 3, 'Height': 4, 'Weight': 3}])
df = pd.concat([df, new_row], ignore_index=True)

# Write the modified df back to a BytesIO buffer (never touches disk)
buffer = BytesIO()
with pd.ExcelWriter(buffer, engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='SKU', index=False)
    df1.to_excel(writer, sheet_name='DFC', index=False)
buffer.seek(0)

# Now use that buffer anywhere that accepts a file-like object
model(buffer)

dfc
   Type  Number of Copies  Length  Width  Height  Weight
0     1                 2       4      4       4      16
1     2                 3       2      2       2       4
sku
   sku  Length  Width  Height  Weight
0    1       2      2       2       2
1    2       1      1       1       1
2    3       2      2       2       2
3    4       1      1       1       1
4    5       2      2       2       2
5    6       2      2       2       2
Gurobi 12.0.3:Gurobi 12.0.3: optimal solution; objective 30
237 simplex iterations
1 branching node
Solve result: solved
Objective:    30
Solve time (Gurobi): 0.2188 seconds
Packing efficiency: 53.12%

=== Copies used ===
               copy_used.val
index0 index1               
1      2                   1

=== SKU assignments ===
                      sku_in_copy.val
index0 index1 index2                 
1      1      2                     1
2      1      2                     1
3      1      2                     1
4      1      2                